## Bibliotecas

In [18]:
import spacy
import pandas as pd
import re
import nltk
from pathlib import Path
from spacy.matcher import Matcher
from spacy.util import filter_spans
from nltk.corpus import stopwords
from pyvis.network import Network
import IPython


## 1. Dados

In [19]:
RAW = Path('../../data/raw')
INTERIM = Path('../../data/interim')
PROCESSED = Path('../../data/processed')

data = pd.read_csv(RAW / 'cases.csv')
metadata = pd.read_csv(RAW / 'metadata.csv')

In [20]:
# merge e seleção do text
full_data = pd.merge(data, metadata)
full_data = full_data[['case_text', 'gender', 'case_id', 'major_mesh_terms', 'mesh_terms']]

# modelo spaCy
nlp = spacy.load('en_core_web_sm')

nltk.download('stopwords', quiet=True)

/home/giizazul/Unicamp/MC896/.venv/lib/python3.13/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/giizazul/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


True

In [21]:
# salva o dataset mesclado
full_data.to_csv(INTERIM / 'full_data_merged.csv', index=False)

## 2. Captura de Medidas (Regex)

In [22]:
#regex para extrair medidas e unidades
measurement_pattern = re.compile(r'(\d+(?:,\d+)?(?:\.\d+)?)\s*(cm|mm|ng/ml|iu/ml|mg)')

#mais detalhado se a gnt quiser
#measurement_pattern = re.compile(r'(\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*[xX]\s*\d+(?:\.\d+)?)*|\d+(?:\.\d+)?(?:\s*[xX]\s*\d+(?:\.\d+)?)*)\s*(cm|mm|ng/ml|iu/ml|mg)')

measurements = []

# processar todos os textos
for idx, row in full_data.iterrows():
    text = str(row['case_text'])
    case_id = row['case_id']
    for match in measurement_pattern.finditer(text):
        value, unit = match.group(1), match.group(2)
        measurements.append({
            'case_id': case_id,
            'node_type': 'VALUE',
            'label_original': match.group(0),
            'label_normalizado': f"{value} {unit}",
            'token_start': -1,
            'token_end': -1,
            'span_start': match.start(),
            'span_end': match.end(),
            'value': value,
            'unit': unit
        })

measurements_df = pd.DataFrame(measurements)

#salva as medidas como arquivos intermediários
measurements_df.to_csv(INTERIM / 'measurements_extracted.csv', index=False)

display(measurements_df.head())

,case_id,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,PMC5137649_01,VALUE,6cm,6 cm,-1,-1,337,340,6,cm
1,PMC5137649_01,VALUE,6cm,6 cm,-1,-1,516,519,6,cm
2,PMC5137649_01,VALUE,9cm,9 cm,-1,-1,522,525,9,cm
3,PMC5137649_01,VALUE,"12,476.5ng/ml","12,476.5 ng/ml",-1,-1,777,790,"12,476.5",ng/ml
4,PMC5137649_01,VALUE,6iu/ml,6 iu/ml,-1,-1,837,843,6,iu/ml


## 3. Extração de Entidades (scispaCy + Matcher) e Unificação de Nós

### Identificar paciente, conceitos e exames

In [23]:
# Listas de palavras-chave atualizadas para se adequarem aos novos tipos de nós
symptom_keywords = [
    "pain", "nausea", "constipation", "bleeding", "fever", "chills", "fatigue", 
    "weakness", "dizziness", "vertigo", "syncope", "headache", "migraine", 
    "dyspnea", "shortness of breath", "cough", "wheezing", "apnea", 
    "diarrhea", "vomiting", "dysphagia", "itching", "swelling", "bruise", "myalgia"
]

finding_keywords = [
    "malignancy", "neoplasm", "cyst", "lesion", "abnormality", "tumor", "mass", 
    "nodule", "polyp", "ulcer", "inflammation", "infection", "abscess", 
    "metastasis", "rash", "erythema", "pruritus", "eczema", "dermatitis", 
    "edema", "effusion", "ascites", "hematoma", "contusion", "laceration", 
    "wound", "scar", "fibrosis", "sclerosis", "atrophy", "hypertrophy", 
    "hyperplasia", "dysplasia", "defect", "deformity", "anomaly"
]

diagnosis_keywords = [
    "carcinoma", "adenocarcinoma", "melanoma", "sarcoma", "lymphoma", "leukemia", 
    "cancer", "sepsis", "pneumonia", "asthma", "copd", "emphysema", "bronchitis", 
    "tuberculosis", "hypertension", "hypotension", "infarction", "ischemia", 
    "arrhythmia", "tachycardia", "bradycardia", "fibrillation", "heart failure", 
    "stroke", "aneurysm", "thrombosis", "embolism", "hemorrhage", "anemia", 
    "leukopenia", "thrombocytopenia", "coagulopathy", "diabetes", "hypothyroidism", 
    "hyperthyroidism", "obesity", "malnutrition", "anorexia", "cirrhosis", 
    "hepatitis", "jaundice", "pancreatitis", "appendicitis", "cholecystitis", 
    "gallstones", "obstruction", "perforation", "rupture", "fistula", "stricture", 
    "hernia", "ileus", "arthritis", "osteoarthritis", "rheumatoid arthritis", 
    "osteoporosis", "fracture", "dislocation", "sprain", "strain", "neuropathy", 
    "neuralgia", "seizure", "epilepsy", "dementia", "alzheimer", "parkinson", 
    "tremor", "paralysis", "paresis", "aphasia", "depression", "anxiety", 
    "schizophrenia", "bipolar disorder", "insomnia", "psoriasis", "cellulitis", 
    "burn", "syndrome", "disorder", "disease", "condition", "failure", 
    "insufficiency", "dysfunction", "impairment", "allergies", "anaphylaxis", 
    "hypoxia", "hypoxemia", "acidosis", "alkalosis", "hyperkalemia", "hypokalemia", 
    "hyponatremia", "hypernatremia", "dehydration"
]


In [24]:
#lista de termos relacionados a exames médicos
exam_keywords = [
    "computed tomography", "ct", "ct scan", "cat scan", "magnetic resonance imaging", 
    "mri", "ultrasound", "sonogram", "echography", "x-ray", "radiograph", 
    "radiography", "pet scan", "positron emission tomography", "fluoroscopy", 
    "angiogram", "angiography", "mammogram", "mammography", "dexa scan", 
    "bone density scan", "electrocardiogram", "ecg", "ekg", "echocardiogram", 
    "echo", "electroencephalogram", "eeg", "electromyography", "emg", 
    "endoscopy", "colonoscopy", "sigmoidoscopy", "gastroscopy", 
    "oesophagogastroduodenoscopy", "bronchoscopy", "laryngoscopy", "cystoscopy", 
    "arthroscopy", "laparoscopy", "colposcopy", "hysteroscopy", "biopsy", 
    "fine needle aspiration", "fna", "lumbar puncture", "spinal tap", 
    "bone marrow aspiration", "bone marrow biopsy", "blood test", "complete blood count", 
    "cbc", "basic metabolic panel", "bmp", "comprehensive metabolic panel", "cmp", 
    "lipid panel", "liver function test", "lft", "thyroid function test", "tft", 
    "urinalysis", "urine culture", "blood culture", "stool test", "pcr", 
    "serology", "coagulation profile", "pt", "aptt", "inr", "d-dimer", 
    "troponin", "creatinine", "bun", "hba1c", "glucose test", "pap smear", 
    "amniocentesis", "spirometry", "pulmonary function test", "pft", "stress test", 
    "holter monitor", "tilt table test", "allergy test", "patch test", "skin prick test", 
    "resection", "pancreatectomy", "appendectomy", "cholecystectomy", "mastectomy", 
    "lumpectomy", "thyroidectomy", "prostatectomy", "hysterectomy", "oophorectomy", 
    "salpingectomy", "craniotomy", "lobectomy", "pneumonectomy", "gastrectomy", 
    "colectomy", "splenectomy", "nephrectomy", "cystectomy", "herniorrhaphy", 
    "arthroplasty", "osteotomy", "ligation", "bypass", "graft", "transplant", 
    "exploration", "examination", "physical examination", "palpation", "auscultation", 
    "percussion", "pathology", "cytology", "histology", "autopsy", "necropsy", 
    "smear", "swab", "culture", "karyotype", "genetic test", "sequencing", 
    "microarray", "flow cytometry", "chromatography", "spectrometry", "assay", 
    "titration", "dialysis", "hemodialysis", "plasmapheresis", "transfusion",
    "catheterization", "intubation", "ventilation", "defibrillation", "cardioversion",
    "pacemaker insertion", "stent placement", "angioplasty", "thrombectomy", 
    "embolectomy", "endarterectomy", "amputation", "debridement", "incision", 
    "drainage", "suturing", "cauterization", "laser therapy", "radiotherapy", 
    "chemotherapy", "immunotherapy", "physical therapy", "occupational therapy",
    "audiometry", "tympanometry", "fundoscopy", "tonometry", "visual acuity test",
    "neurological examination", "mental status examination", "glasgow coma scale",
    "apgar score", "treadmill test", "barium swallow", "barium enema", "dexa",
    "scintigraphy", "spect scan", "doppler", "elastography", "polysomnography",
    "sleep study", "sweat test", "mantoux test", "tuberculin skin test", "scan"
]

In [25]:
patient_terms = ["woman", "man", "girl", "boy", "male", "female", "patient"]

In [26]:
stop_words_nltk = set(stopwords.words('english'))
custom_stops = {'day', 'history', 'month', 'year', 'time', 'presence', 'evidence', 'fig', 'figure'}
all_stopwords = stop_words_nltk.union(custom_stops)

In [27]:
negation_words = {"no", "not", "deny", "denies", "without", "negative", "absence"}
generic_standalone_words = {"level", "cm x", "figs", "portion", "activity", "return", "follow", "cm", "mm", "ml", "mg", "ca", "cm cyst", "cyst cm", "diagnosis"}

In [28]:
patient_matcher = Matcher(nlp.vocab)
patient_matcher.add("PATIENT_DEMO", [[{"LOWER": {"IN": patient_terms}}]])

concept_matcher = Matcher(nlp.vocab)
#padrão: adjetivo(0+) seguido de substantivo(1+)
concept_matcher.add("CLINICAL_CONCEPT", [
    [{"POS": "ADJ", "OP": "*"}, {"POS": "NOUN", "OP": "+"}],
    [{"POS": "PROPN", "OP": "+"}]
])

In [29]:
extracted_entities = []

# Loop sobre todos os textos
for idx, row in full_data.iterrows():
    text = str(row['case_text'])
    case_id = row['case_id']
    doc = nlp(text)
    
    patient_label = "patient"
    patient_matches = patient_matcher(doc)
    if patient_matches:
        _, start, end = patient_matches[0]
        span = doc[start:end]
        patient_label = span.text.lower()
        extracted_entities.append({
            'case_id': case_id,
            'node_type': 'PATIENT',
            'label_original': span.text,
            'label_normalizado': patient_label,
            'token_start': start, 'token_end': end,
            'span_start': span.start_char, 'span_end': span.end_char,
            'value': None, 'unit': None
        })

    #filtra sobreposições
    spans = [doc[start:end] for _, start, end in concept_matcher(doc)]
    filtered_spans = filter_spans(spans)

    #extrai entidades e classifica com base em palavras-chave
    for span in filtered_spans:

        # filtra tokens irrelevantes (stopwords, pontuação, dígitos)
        valid_tokens = [
            t for t in span 
            if t.lemma_.lower() not in all_stopwords 
            and not t.is_punct 
            and not t.is_digit
        ]

        if not valid_tokens: 
            continue

        clean_label = " ".join([t.lemma_.lower() for t in valid_tokens])

        if patient_label in clean_label: 
            continue
        
        start_token = span.start
        preceding_tokens = [t.lemma_.lower() for t in doc[max(0, start_token - 3):start_token]]
        if any(neg in preceding_tokens for neg in negation_words): continue
        if clean_label in generic_standalone_words: continue

        # classificação baseada nas listas de palavras-chave (doenças e exames)
        
        node_class = 'FINDING'
        if any(word in clean_label for word in symptom_keywords): 
            node_class = 'SYMPTOM'
        elif any(word in clean_label for word in diagnosis_keywords): 
            node_class = 'DIAGNOSIS'
        elif any(word in clean_label for word in finding_keywords): 
            node_class = 'FINDING'
        elif any(word in clean_label for word in exam_keywords): 
            node_class = 'PROCEDURE'

        
        extracted_entities.append({
            'case_id': case_id,
            'node_type': node_class,
            'label_original': span.text,
            'label_normalizado': span.lemma_.lower(),
            'token_start': span.start, 'token_end': span.end,
            'span_start': span.start_char, 'span_end': span.end_char,
            'value': None, 'unit': None
        })

entities_df = pd.DataFrame(extracted_entities).drop_duplicates(subset=['case_id', 'label_normalizado']).reset_index(drop=True)
entities_df.to_csv(INTERIM / 'entities_extracted.csv', index=False)
display(entities_df.head())

,case_id,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,PMC5137649_01,PATIENT,woman,woman,6,7,14,19,None,None
1,PMC5137649_01,FINDING,right flank,right flank,15,17,54,65,None,None
2,PMC5137649_01,SYMPTOM,lower quadrant abdominal pain,low quadrant abdominal pain,18,22,70,99,None,None
3,PMC5137649_01,SYMPTOM,nausea,nausea,24,25,116,122,None,None
4,PMC5137649_01,SYMPTOM,constipation,constipation,26,27,127,139,None,None


### Tabela de Nós Final

In [30]:
#concatena as entidades e medidas extraídas em um único dataframe de nós
final_nodes_df = pd.concat([entities_df, measurements_df], ignore_index=True)
final_nodes_df.insert(0, 'node_id', [f"N_{i+1:03d}" for i in range(len(final_nodes_df))])

#exporta os nós como arquivo final
final_nodes_df.to_csv(PROCESSED / 'nodes.csv', index=False)

print("TABELA DE NÓS")
display(final_nodes_df.head())

TABELA DE NÓS


,node_id,case_id,node_type,label_original,label_normalizado,token_start,token_end,span_start,span_end,value,unit
0,N_001,PMC5137649_01,PATIENT,woman,woman,6,7,14,19,None,None
1,N_002,PMC5137649_01,FINDING,right flank,right flank,15,17,54,65,None,None
2,N_003,PMC5137649_01,SYMPTOM,lower quadrant abdominal pain,low quadrant abdominal pain,18,22,70,99,None,None
3,N_004,PMC5137649_01,SYMPTOM,nausea,nausea,24,25,116,122,None,None
4,N_005,PMC5137649_01,SYMPTOM,constipation,constipation,26,27,127,139,None,None


## 4. Geração do Grafo de Conhecimento (Tabela de Arestas)

In [31]:
#define relações entre verbos e tipos de arestas no grafo
verb_relations = {
    ("present", "have", "experience", "associate"): "HAS_SYMPTOM",
    ("undergo", "perform", "do", "plan", "convert"): "UNDERWENT_PROCEDURE",
    ("reveal", "demonstrate", "suggest", "show", "note", "observe"): "SUPPORTS",
    ("treat", "resect", "discharge"): "TREATED_BY"
}

### Arestas Sintáticas e Contexto 

In [32]:
edges = []

for case_id, group in final_nodes_df.groupby('case_id'):
    # Precisamos do texto para esse case_id para buscar as sentenças e o contexto
    text = str(full_data[full_data['case_id'] == case_id]['case_text'].values[0])
    doc = nlp(text)
    
    patient_node = group[group['node_type'] == 'PATIENT']
    
    # Se não houver nó de paciente na tabela, definimos valores padrão
    if not patient_node.empty:
        patient_label = patient_node['label_normalizado'].values[0]
        patient_text = patient_node['label_original'].values[0]
        patient_start = int(patient_node['span_start'].values[0])
        patient_end = int(patient_node['span_end'].values[0])
    else:
        patient_label = 'patient'
        patient_text = 'patient'
        patient_start = -1
        patient_end = -1
        
    patient_type = 'PATIENT'

    # criação das arestas com base nos verbos e entidades extraídas
    for sent in doc.sents:
        nodes_in_sent = [row for _, row in group.iterrows() if row['node_type'] != 'PATIENT' and sent.start <= row['token_start'] < sent.end]
        
        for token in sent:
            if token.pos_ == "VERB":
                verbo_lema = token.lemma_.lower()

                for verbs, rel in verb_relations.items():
                    if verbo_lema in verbs:
                        if rel in ["HAS_SYMPTOM", "UNDERWENT_PROCEDURE", "TREATED_BY"]:
                            for node in nodes_in_sent:
                                # filtra um pouco
                                if rel == "HAS_SYMPTOM" and node['node_type'] not in ['SYMPTOM']: continue
                                if rel == "UNDERWENT_PROCEDURE" and node['node_type'] not in ['EXAM']: continue
                                
                                # Somente o texto entre as duas entidades
                                if patient_end != -1 and patient_end <= node['span_start']:
                                    # Paciente vem antes do nó
                                    inicio = patient_end
                                    fim = node['span_start']
                                elif patient_start != -1 and node['span_end'] <= patient_start:
                                    # Nó vem antes do paciente
                                    inicio = node['span_end']
                                    fim = patient_start
                                else:
                                    # Caso de sobreposição ou sem paciente detectado, pega do começo da frase até o nó (fallback)
                                    inicio = min(sent.start_char, node['span_start'])
                                    fim = max(sent.start_char, node['span_start'])
                                    
                                contexto = text[inicio:fim].strip()
                                
                                edges.append({
                                    'case_id': case_id,
                                    
                                    'e1_normalizado': patient_label,
                                    'e1_text': patient_text,
                                    'e1_type': patient_type,
                                    
                                    'e2_normalizado': node['label_normalizado'],
                                    'e2_text': node['label_original'],
                                    'e2_type': node['node_type'],
                                    
                                    'relation_label': rel,
                                    'intermediate_text': contexto
                                })
                                
                        elif rel == "SUPPORTS" and len(nodes_in_sent) >= 2:
                            n1 = nodes_in_sent[0]
                            n2 = nodes_in_sent[1]
                            
                            # Somente o texto entre as duas entidades
                            inicio = min(n1['span_end'], n2['span_end'])
                            fim = max(n1['span_start'], n2['span_start'])
                            
                            if inicio > fim:
                                contexto = ""
                            else:
                                contexto = text[inicio:fim].strip()
                            
                            edges.append({
                                'case_id': case_id,
                                
                                'e1_normalizado': n1['label_normalizado'],
                                'e1_text': n1['label_original'],
                                'e1_type': n1['node_type'],
                                
                                'e2_normalizado': n2['label_normalizado'],
                                'e2_text': n2['label_original'],
                                'e2_type': n2['node_type'],
                                
                                'relation_label': rel,
                                'intermediate_text': contexto
                            })
                        break

edges_df = pd.DataFrame(edges)
display(edges_df.head())


,case_id,e1_normalizado,e1_text,e1_type,e2_normalizado,e2_text,e2_type,relation_label,intermediate_text
0,PMC10106591_01,female,female,PATIENT,intermittent chest pain,intermittent chest pain,SYMPTOM,HAS_SYMPTOM,patient presented to the emergency room (ER) o...
1,PMC10106591_01,physical examination,Physical examination,PROCEDURE,admission,admission,FINDING,SUPPORTS,upon
2,PMC10106591_01,pulmonary examination,pulmonary examination,PROCEDURE,normal heart,normal heart,FINDING,SUPPORTS,revealed
3,PMC10106591_01,electrocardiogram,electrocardiogram,PROCEDURE,ecg,ECG,PROCEDURE,SUPPORTS,(
4,PMC10106591_01,electrocardiogram,electrocardiogram,PROCEDURE,ecg,ECG,PROCEDURE,SUPPORTS,(


### Arestas de Valores por distância de caracteres 

In [33]:
for case_id, group in final_nodes_df.groupby('case_id'):
    text = str(full_data[full_data['case_id'] == case_id]['case_text'].values[0])
    meas_group = group[group['node_type'] == 'VALUE']
    ent_group = group[(group['node_type'] != 'VALUE') & (group['node_type'] != 'PATIENT')]
    
    for _, meas_row in meas_group.iterrows():
        closest_entity = None
        min_gap = 30

        #criação das arestas entre entidades e medidas com base na proximidade no texto
        for _, ent_row in ent_group.iterrows():
            if ent_row['span_end'] <= meas_row['span_start']:
                gap = meas_row['span_start'] - ent_row['span_end']
            elif meas_row['span_end'] <= ent_row['span_start']:
                gap = ent_row['span_start'] - meas_row['span_end']
                if gap > 2: gap += 100
            else:
                gap = 0
            
            if gap < min_gap:
                min_gap = gap
                closest_entity = ent_row
                
        if closest_entity is not None:
            inicio = min(closest_entity['span_end'], meas_row['span_end'])
            fim = max(closest_entity['span_start'], meas_row['span_start'])
            contexto = text[inicio:fim].strip()
            
            # ADAPTADO PARA O NOVO PADRÃO
            edges.append({
                'case_id': case_id,
                
                'e1_normalizado': closest_entity['label_normalizado'],
                'e1_text': closest_entity['label_original'],
                'e1_type': closest_entity['node_type'],
                
                'e2_normalizado': meas_row['label_normalizado'],
                'e2_text': meas_row['label_original'],
                'e2_type': meas_row['node_type'],
                
                'relation_label': 'HAS_VALUE',
                'intermediate_text': contexto
            })


## Grafo de Conhecimento final

In [34]:
# Removemos duplicatas usando a nova nomenclatura
final_graph_edges_df = pd.DataFrame(edges).drop_duplicates(subset=['case_id', 'e1_normalizado', 'e2_normalizado', 'relation_label']).reset_index(drop=True)

if 'edge_id' not in final_graph_edges_df.columns:
    final_graph_edges_df.insert(0, 'edge_id', [f"E_{i+1:03d}" for i in range(len(final_graph_edges_df))])

final_graph_edges_df.to_csv(PROCESSED / 'edges.csv', index=False)
display(final_graph_edges_df.head())


,edge_id,case_id,e1_normalizado,e1_text,e1_type,e2_normalizado,e2_text,e2_type,relation_label,intermediate_text
0,E_001,PMC10106591_01,female,female,PATIENT,intermittent chest pain,intermittent chest pain,SYMPTOM,HAS_SYMPTOM,patient presented to the emergency room (ER) o...
1,E_002,PMC10106591_01,physical examination,Physical examination,PROCEDURE,admission,admission,FINDING,SUPPORTS,upon
2,E_003,PMC10106591_01,pulmonary examination,pulmonary examination,PROCEDURE,normal heart,normal heart,FINDING,SUPPORTS,revealed
3,E_004,PMC10106591_01,electrocardiogram,electrocardiogram,PROCEDURE,ecg,ECG,PROCEDURE,SUPPORTS,(
4,E_005,PMC10106591_01,acute coronary syndrome,acute coronary syndrome,DIAGNOSIS,acs,ACS,FINDING,SUPPORTS,(
